# VecDB Bulk Loading – BYO Vectors (Auto IDs)

## 1. Scenario Overview
Use this path when your CSV supplies vectors but omits IDs. The table is created with `auto_generate_id=true`, letting VecDB assign surrogate IDs during ingestion.

Upload `bulk_loading/data/bulktable_byov_auto_ids.csv` to Object Storage and store the signed URL in `BYO_AUTO_CSV_URL`.


## 2. Setup





### Install Required Packages
Run the following cell once per environment to install / upgrade `oracle-vecdb`, `python-dotenv`, and `pandas`. Skip this step if you already have the dependencies available in your kernel.



In [ ]:
%pip install -U oracle-vecdb python-dotenv pandas



### Load Credentials and Configure Demo Inputs
This cell reads `.env` for `VECDB_REST_URL`, `VECDB_USERNAME`/`VECDB_PASSWORD`, wires up the published Object Storage URLs, and defines table names plus the embedding model used later. Update the `.env` file or override any env var before running.



In [ ]:
import os
from dotenv import load_dotenv
from oracle_vecdb import OracleVecDB, Configuration

print('Loading VecDB environment variables...')
load_dotenv()

resolved_host = os.getenv('VECDB_REST_URL')
resolved_user = os.getenv('VECDB_USERNAME')
resolved_password = os.getenv('VECDB_PASSWORD')
resolved_access_token = os.getenv('VECDB_ACCESS_TOKEN')
print(f'Resolved REST endpoint: {resolved_host}')
print(f'Resolved user: {resolved_user}')

config_kwargs = {"rest_url": resolved_host}
if resolved_access_token:
    config_kwargs["access_token"] = resolved_access_token
else:
    config_kwargs["username"] = resolved_user
    config_kwargs["password"] = resolved_password
config = Configuration(**config_kwargs)
if os.getenv('VECDB_SELF_SIGNED_SSL', 'false').lower() == 'true':
    config.verify_ssl = False
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    print('Disabled SSL verification (self-signed certificates).')

vecdb = OracleVecDB(config)
auth_method = 'bearer token' if config.access_token else 'username/password'
print('Connected to', config.rest_url)
print('Auth method:', auth_method)
print('VecDB client ready for this workflow.')

BYO_AUTO_CSV_URL = os.getenv('BYO_AUTO_CSV_URL')
if not BYO_AUTO_CSV_URL:
    raise RuntimeError('Set BYO_AUTO_CSV_URL with the Object Storage URL for this scenario.')
print('BYO auto-ID CSV URL:', BYO_AUTO_CSV_URL)

BYO_TABLE_AUTO = os.getenv('BYO_TABLE_AUTO')
print('BYO auto-ID table:', BYO_TABLE_AUTO)


## 11. Bulk load vectors and auto generate record ids
Create a second BYO table where VecDB should assign IDs during ingest. To fully exercise this mode, upload a CSV that omits the `ID` column, host it in Object Storage, and point the environment variable to that signed URL before running the load step.


In [ ]:

print('Resetting BYO vectors table (auto-generated IDs)...')
try:
    vecdb.drop_vector_table(name=BYO_TABLE_AUTO)
    print(f'Dropped existing table {BYO_TABLE_AUTO}')
except Exception:
    print(f'Table {BYO_TABLE_AUTO} not present; creating new table')

vecdb.create_vector_table(
    name=BYO_TABLE_AUTO,
    comment='Support tickets demo table with BYO vectors (auto IDs)',
    annotations={'DATASET': 'support_tickets'},
    table_params={"auto_generate_id":True},
)
print('Ready for BYO auto-ID load:', BYO_TABLE_AUTO)



## 12. Bulk Load BYO Dataset (auto_generate_id=true)
Submit the auto-ID load job. If the backing CSV lacks IDs, VecDB will generate new GUIDs; if IDs exist in the file, VecDB preserves them (as currently seen in the shared dataset).



In [ ]:
print('Submitting BYO auto-ID bulk load job...')
byo_auto_job = vecdb.load_vectors(
    table_name=BYO_TABLE_AUTO,
    url=BYO_AUTO_CSV_URL,
)
BYO_AUTO_JOB_NAME = getattr(byo_auto_job, 'job_name', None)
if BYO_AUTO_JOB_NAME:
    print('BYO auto-ID load job name:', BYO_AUTO_JOB_NAME)
else:
    print('Raw BYO auto-ID load job response:', byo_auto_job)



## 13. Inspect BYO Auto-ID Load Job
Monitor the auto-ID job using the same describe/log APIs. This ensures teams can validate ingest behavior regardless of how IDs are sourced.



In [ ]:
job_name = globals().get('BYO_AUTO_JOB_NAME')
if job_name:
    details = vecdb.describe_vector_load_job(load_job_name=job_name)
    print('BYO auto-ID job details:', details)
    try:
        logs = vecdb.get_vector_load_job_log(load_job_name=job_name)
        print('BYO auto-ID log preview:', str(logs)[:500])
    except Exception as exc:
        print('Unable to fetch BYO auto-ID load log:', exc)
else:
    print('Run the BYO auto-ID load cell first to capture a job identifier.')



## 14. List BYO Auto-ID Vectors
List a few rows from `BYO_TABLE_AUTO`. Because the CSV omits IDs, expect VecDB-generated identifiers in the results.


In [ ]:

byo_auto_vectors = vecdb.list_vectors(table_name=BYO_TABLE_AUTO, limit=5)
print('Listed vectors count:', len(byo_auto_vectors.items))
print('Sample BYO auto-ID vectors:', byo_auto_vectors)
for item in byo_auto_vectors.items:
    print(item.id, item.metadata, item.dense_vector)



## Cleanup
Drop the demo table so repeated runs stay isolated.

In [ ]:
print('Dropping BYO auto-ID table...')
try:
    vecdb.drop_vector_table(name=BYO_TABLE_AUTO)
    print('Dropped', BYO_TABLE_AUTO)
except Exception as exc:
    print('Unable to drop', BYO_TABLE_AUTO, exc)